# Vibrations Explorer
Interactive companion to TU Delft AE2135-II Vibrations. Run all cells, then move the sliders.

*Aaron James Brouns*

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact

## Helper: simulate a mass-spring-damper

Every SDOF example below solves the same equation of motion

$$m\ddot{x} + c\dot{x} + kx = F(t)$$

numerically. We write it as two first order equations, with state $[x, \dot{x}]$, and let `solve_ivp` integrate it.

In [5]:
def simulate(m, c, k, x0=0.0, v0=0.0, force=lambda t: 0.0, t_end=10.0):
    def rhs(t, y):
        x, v = y
        a = (force(t) - c * v - k * x) / m      # Newton: m*a = F - c*v - k*x
        return [v, a]

    t = np.linspace(0, t_end, 2000)
    sol = solve_ivp(rhs, (0, t_end), [x0, v0], t_eval=t)
    return sol.t, sol.y[0]

## 1. Free vibration

With no force, the response depends on two numbers:

$$\omega_n = \sqrt{k/m}, \qquad \zeta = \frac{c}{2\sqrt{km}}$$

- $\zeta < 1$: underdamped, the mass oscillates and decays
- $\zeta = 1$: critically damped, fastest return without overshoot
- $\zeta > 1$: overdamped, slow return without oscillation

**Try:** move $\zeta$ from 0 to 2 and watch the response change character.

In [7]:
@interact(zeta=(0.0, 2.0, 0.05), k=(10, 200, 10), m=(0.5, 5.0, 0.5))
def free_vibration(zeta=0.1, k=50, m=1.0):
    c = 2 * zeta * np.sqrt(k * m)
    t, x = simulate(m, c, k, x0=1.0)

    plt.figure(figsize=(8, 3.5))
    plt.plot(t, x)
    plt.axhline(0, color="k", lw=0.8)
    plt.xlabel("t [s]")
    plt.ylabel("x [m]")
    plt.title(f"ωn = {np.sqrt(k/m):.2f} rad/s,  ζ = {zeta:.2f}")
    plt.grid(alpha=0.3)
    plt.show()

interactive(children=(FloatSlider(value=0.1, description='zeta', max=2.0, step=0.05), IntSlider(value=50, desc…

## 2. Forced vibration and resonance

Now apply a harmonic force $F(t) = F_0\cos\Omega t$. After the transient dies out, the mass vibrates at the forcing frequency with amplitude

$$\frac{X}{F_0/k} = \frac{1}{\sqrt{(1-r^2)^2 + (2\zeta r)^2}}, \qquad r = \frac{\Omega}{\omega_n}$$

**Try:** set $r = 1$ with small $\zeta$ to see resonance. Then try $r = 3$ and see the mass barely move.

In [ ]:
@interact(r=(0.1, 3.0, 0.05), zeta=(0.02, 1.0, 0.02))
def forced_vibration(r=0.9, zeta=0.05):
    m, k = 1.0, 40.0
    wn = np.sqrt(k / m)
    c = 2 * zeta * np.sqrt(k * m)
    Omega = r * wn
    F0 = k                      # chosen so the static deflection F0/k = 1

    t, x = simulate(m, c, k, force=lambda t: F0 * np.cos(Omega * t), t_end=30)

    r_range = np.linspace(0, 3, 500)
    magnification = 1 / np.sqrt((1 - r_range**2)**2 + (2 * zeta * r_range)**2)
    current = 1 / np.sqrt((1 - r**2)**2 + (2 * zeta * r)**2)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    ax1.plot(t, x)
    ax1.set_xlabel("t [s]")
    ax1.set_ylabel("x / (F0/k)")
    ax1.set_title("Time response")

    ax2.plot(r_range, magnification)
    ax2.plot(r, current, "ro")
    ax2.set_xlabel("r = Ω/ωn")
    ax2.set_ylabel("X / (F0/k)")
    ax2.set_title(f"Magnification = {current:.2f}")
    for ax in (ax1, ax2):
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=0.9, description='r', max=3.0, min=0.1, step=0.05), FloatSlider(value=…

## 3. Two degree of freedom system: mode shapes

Two masses connected by three springs: wall, $k_1$, $m_1$, $k_2$, $m_2$, $k_3$, wall.

$$\mathbf{M}\ddot{\mathbf{x}} + \mathbf{K}\mathbf{x} = \mathbf{0}, \qquad
\mathbf{M} = \begin{bmatrix} m_1 & 0 \\ 0 & m_2 \end{bmatrix}, \quad
\mathbf{K} = \begin{bmatrix} k_1+k_2 & -k_2 \\ -k_2 & k_2+k_3 \end{bmatrix}$$

Solving $(\mathbf{K} - \omega^2\mathbf{M})\boldsymbol{\phi} = \mathbf{0}$ gives two natural frequencies and two mode shapes.

**Try:** with equal masses and springs, mode 1 moves both masses together and mode 2 moves them in opposite directions. Then make $m_2$ much heavier.

In [ ]:
from scipy.linalg import eigh

@interact(m1=(0.5, 5.0, 0.5), m2=(0.5, 5.0, 0.5), k1=(10, 100, 10), k2=(10, 100, 10), k3=(10, 100, 10))
def two_dof_modes(m1=1.0, m2=1.0, k1=50, k2=50, k3=50):
    M = np.array([[m1, 0], [0, m2]])
    K = np.array([[k1 + k2, -k2], [-k2, k2 + k3]])

    omega_sq, modes = eigh(K, M)     # solves K φ = ω² M φ
    omega = np.sqrt(omega_sq)

    plt.figure(figsize=(6, 3.5))
    for i in range(2):
        shape = modes[:, i] / np.max(np.abs(modes[:, i]))   # scale so largest value is 1
        plt.plot([1, 2], shape, "o-", ms=10, label=f"Mode {i+1}: ω = {omega[i]:.2f} rad/s")
    plt.axhline(0, color="k", lw=0.8)
    plt.xticks([1, 2], ["mass 1", "mass 2"])
    plt.ylabel("relative displacement")
    plt.ylim(-1.3, 1.3)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

interactive(children=(FloatSlider(value=1.0, description='m1', max=5.0, min=0.5, step=0.5), FloatSlider(value=…